In [ ]:
!pip install -r .\requirements.txt

# Modele

### Chat Model
Jeżeli nie masz pobranego modelu, wykonaj kod !ollama pul... aby pobrać model z .env

#### Non-Streaming

In [ ]:
from app.core import LLM_MODEL
!ollama pull {LLM_MODEL}

In [ ]:
from app.core import ChatModel, LLM_MODEL

chat = ChatModel(model=LLM_MODEL, system="You are a helpful assistant", memory=False)
r = chat.ask("Napisz małą rozprawkę o Szekspirze", think=False)

print(r)

#### Streaming
W .ipnyb streaming nie zadziała

In [ ]:
from app.core import ChatModel, LLM_MODEL

chat = ChatModel(model=LLM_MODEL, system="You are a helpful assistant", memory=False)
for piece in chat.ask_stream("Napisz małą rozprawkę o Szekspirze", think=False):
    print(piece, end="", flush=True)

### Embed Model
Jeżeli nie masz pobranego modelu, wykonaj kod !ollama pul... aby pobrać model z .env

In [ ]:
from app.core import EMBED_MODEL
!ollama pull {EMBED_MODEL}

In [ ]:
from app.core import EmbedModel, EMBED_MODEL, EMBED_MODEL_DIM

embed = EmbedModel(model=EMBED_MODEL, dim=EMBED_MODEL_DIM)
e = embed.encode("Lorem ipsum dolor sit amed")

print(e)

# RAG
Stwórz bazy danych tą komendą jeżeli jeszcze tego nie zrobiłeś

In [ ]:
!docker compose -f database/docker-compose.yml --env-file .env up -d

### Graph RAG

#### Procedural Graph Database Set-Up

#### LLM Graph Database Set-Up

In [ ]:
from app.core import GRAPH_MODEL
!ollama pull {GRAPH_MODEL}

In [12]:
from app.graph import initialize_knowledge_graph, initialize_graph_driver, purge_database, graph_driver

initialize_graph_driver()
purge_database(graph_driver)

initialize_knowledge_graph()

OK: Połączenie z neo4j działa
APOC jest dostępne.


In [13]:
from app.ingest import load_knowledge, check_for_duplicates
from app.schema import prepare_for_vector_embedding, prepare_for_lexical_search, prepare_for_prompt

documents = load_knowledge("./knowledge")
check_for_duplicates(documents)

document_string: str = '\n\n'.join(
    [document.__repr__() for document in documents]
)

print(document_string)

{'id': 'proc.magazyn.przyjecie-pz', 'title': 'Przyjęcie towaru na magazyn dokumentem PZ', 'module': 'magazyn', 'summary': 'Jak przyjąć towar od dostawcy na wybrany magazyn i zatwierdzić dokument.', 'query': ['jak przyjąć towar', 'towar przyjechał co teraz', 'jak zrobić pz', 'jak zwiększyć stan magazynowy', 'przyjęcie zewnętrzne'], 'preconditions': [], 'roles': ['magazynier', 'kierownik'], 'steps': [{'text': 'Przejdź do Dokumenty w menu bocznym', 'anchor': 'nav.documents'}, {'text': 'Kliknij Nowy dokument', 'anchor': 'btn.document-new'}, {'text': 'W polu Typ dokumentu wybierz PZ', 'anchor': 'field.document-type', 'action': {'kind': 'select', 'label': 'PZ'}, 'note': 'Po zapisaniu typu nie da się zmienić'}, {'text': 'W polu Dostawca wybierz kontrahenta, od którego przyjmujesz towar', 'anchor': 'field.counterparty', 'action': {'kind': 'select', 'label': 'Stalmex'}}, {'text': 'W sekcji Magazyny wybierz Magazyn docelowy', 'anchor': 'field.warehouse-to', 'action': {'kind': 'select', 'label': 

In [15]:
from app.graph import build_graph_with_ollama, print_graph, knowledge_graph, graph_driver
from app.core import GRAPH_MODEL

build_graph_with_ollama(model='qwen3.5:35b', documents=document_string)

print_graph()

input("[ENTER], aby zsynchronizować do bazy danych...")

knowledge_graph.sync(driver=graph_driver)

print("\nGotowe!")

I need to transform this documentation into a structured knowledge graph. Let me analyze what I have:

1. **Procedures** (procedury):
   - proc.magazyn.przyjecie-pz: Przyjęcie towaru na magazyn dokumentem PZ
   - proc.magazyn.wydanie-wz: Wydanie towaru z magazynu dokumentem WZ
   - proc.magazyn.przesuniecie-mm: Przesunięcie towaru między magazynami dokumentem MM
   - proc.magazyn.sprawdzenie-stanu: Sprawdzenie stanu magazynowego produktu

2. **Errors** (błędy):
   - ERR-1003: Nie można zatwierdzić dokumentu bez pozycji
   - ERR-1004: Na magazynie źródłowym nie ma wystarczającej ilości produktu
   - ERR-1005: Przesunięcie MM musi mieć różne magazyny
   - ERR-3001: Tylko kierownik może zatwierdzać przesunięcia

3. **Concepts** (koncepcje):
   - concept.dokument-magazynowy: Dokumenty PZ, WZ, MM
   - concept.stan-magazynowy: Stan magazynowy
   - concept.role-magazynowe: Role użytkowników w module

First, I need to check what classes and relationships already exist in the graph. Then I'll d

Zapisywanie relacji: 100%|██████████| 20/20 [00:00<00:00, 46.83it/s]


Gotowe!


In [ ]:
!start http://localhost:7474